In [1]:
import os
import torch
from openai import OpenAI
from transformers import GPT2Tokenizer
from dotenv import load_dotenv

d:\Coding\GitHub Repos\Personalized-LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

In [3]:
#RQ1: Step 1 - Setting up a prior belief based on targeted question

#targeted questions
question1 = "Do you prefer direct next steps or detailed discussion first?"
question1_options=["1. direct steps", "2. detailed discussion"]

question2= "Would you prefer a checklist or conversation?"
question2_options=["1. checklist", "2. conversation"]

In [4]:
#Function for asking question

def ask_question(question, answer):
    #question 1
    print(question)
    for i in range(len(answer)):
        print(answer[i])
    choice= input('Enter your preferred answer number') #change it to better UI based button later
    return answer[int(choice)-1]


choice_result = []

choice_result.append(ask_question(question1, question1_options))
choice_result.append(ask_question(question2, question2_options))

print(choice_result)


Do you prefer direct next steps or detailed discussion first?
1. direct steps
2. detailed discussion
Would you prefer a checklist or conversation?
1. checklist
2. conversation
['1. direct steps', '1. checklist']


In [22]:
#creating user profile based on user's answer
from collections import defaultdict
user_profile = defaultdict(list)
user_profile = {"preferred_conversation_style": None,
                "ph1": None,
                "ph2": None,
                "peh1": None,
                "peh2": None,
                "avgpeh1": []}

#take 2 answers and based on that assign key-value pair for user profile--think
if choice_result[0]=="1. direct steps" and choice_result[1] == "1. checklist":
    user_profile["preferred_conversation_style"] = "action_based"
elif choice_result[0]=="2. detailed discussion" and choice_result[1] == "2. conversation":
    user_profile["preferred_conversation_style"] = "relationship_based"
else:
    user_profile["preferred_conversation_style"] = "mixed"
        
print(user_profile)


{'preferred_conversation_style': 'action_based', 'ph1': None, 'ph2': None, 'peh1': None, 'peh2': None, 'avgpeh1': []}


In [23]:
#assign prior belief value - hardcoded, might change with an LLM prompt layer (step 1 completed)

if user_profile["preferred_conversation_style"] == "action_based":
    user_profile["ph1"] = .90
    user_profile["ph2"] = .10
elif user_profile["preferred_conversation_style"] == "relationship_based":
    user_profile["ph1"] = .10
    user_profile["ph2"] = .90
else:
    user_profile["ph1"]=.60
    user_profile["ph2"]=.40
    
print(user_profile)

{'preferred_conversation_style': 'action_based', 'ph1': 0.9, 'ph2': 0.1, 'peh1': None, 'peh2': None, 'avgpeh1': []}


In [24]:
#RQ1: Step 2 - Observing behavior with LLM prompt and treating each message like evidance P(E)

user_input = "Can you elaborate this topic?"                          #input("Hi there! What you want to chat about?")

In [ ]:
#torch.set_default_device("cuda")


In [ ]:
#model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", torch_dtype="auto", trust_remote_code=True)
#tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2", trust_remote_code=True)

#tokenizer = GPT2Tokenizer.from_pretrained("gpt2")


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.56s/it]


In [28]:
def calculate_peh1_value(user_profile):

    # Create full prompt with few-shot examples
    full_prompt = f'''
    You are a conversation analyst. Estimate two probabilities:
    - peh1 = probability (0 to 1) that the user is Action-Based
    - peh2 = probability (0 to 1) that the user is Relationship-Based

    Here are some examples:

    Example 1:
    User: "Tell me the fastest way to finish this project."
    peh1: 0.90
    peh2: 0.10

    Example 2:
    User: "Before we start, can we discuss what's most important to focus on together?"
    peh1: 0.20
    peh2: 0.80

    Example 3:
    User: "I just want to get this over with quickly."
    peh1: 0.85
    peh2: 0.15

    Now, analyze the following user input:

    User: {user_input}

    Output:
    peh1: **float number only**
    peh2: **float number only**
    '''

    # Send the full prompt to OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
    response_text = response.choices[0].message.content
    print(response_text)
    
    for lines in response_text.splitlines():
        if lines.strip().startswith("peh1"):
            user_profile["peh1"] = float(lines.split(":")[1].strip())
            user_profile["avgpeh1"].append(user_profile["peh1"])
        elif lines.strip().startswith("peh2"):
            user_profile["peh2"] = float(lines.split(":")[1].strip())
        
    return user_profile
    
calculate_peh1_value(user_profile)


peh1: 0.40  
peh2: 0.60


{'preferred_conversation_style': 'action_based',
 'ph1': 0.9,
 'ph2': 0.1,
 'peh1': 0.4,
 'peh2': 0.6,
 'avgpeh1': [0.4]}

{'preferred_conversation_style': 'action_based', 'ph1': 0.9, 'ph2': 0.1, 'peh1': 0.3, 'peh2': 0.7}


In [10]:
#using bayesian theorem, calculating posterior 

numerator = user_profile["peh1"] * user_profile["ph1"]
denominator = (user_profile["peh1"] * user_profile["ph1"]) + (user_profile["peh2"] * user_profile["ph2"])

ph1e = numerator/denominator

ph2e = 1 - ph1e

print(ph1e, ph2e)

# after first input
user_profile["ph1"] = ph1e
user_profile["ph2"] = ph2e

0.7941176470588235 0.20588235294117652


In [ ]:
# Create full prompt with few-shot examples
updated_prompt = f'''
You're a Computer Science note maker. 
Generate notes for the user according to their preferred conversation style based on {ph1e} and {ph2e} score.

ph1e means probability the user is action based given what they just said.
ph2e means probability the user is relationship based given what they just said.

if the user is action based, give output in the following format -
1. direct and simple sentence
2. short and concise sentence
3. task focused and in bullet points

if the user is relationship based, give output in the following format -
1. direct and simple sentence
2. storyline conversation
3. focus on emotion and in paragraph style

Now generate notes on Array using python coding language in 300 words. 
'''


**Action-Based Style:**

1. An array is a data structure used to store elements in a specific order.
2. Arrays in Python are managed by importing libraries like 'array', or more commonly, using lists.
3. Task-focused steps:
   - To create an array, use: `import array as arr` and then `a = arr.array('typecode', [elements])`.
   - For lists: `a = [element1, element2, element3, ...]`.
   - Access elements using indices: `a[index]`.
   - Modify elements: `a[index] = new_value`.
   - Add elements: `a.append(new_value)` for lists, or `a.insert(index, new_value)`.
   - Remove elements: `a.remove(value)` or `a.pop(index)`.
   - Loop through arrays: `for element in a:`.

4. Use the NumPy library for more complex operations and benefits of multidimensional arrays.
5. Import NumPy: `import numpy as np`.
6. Create a NumPy array: `np_array = np.array([elements])`.
7. NumPy arrays are faster and more efficient.
8. Learn slicing for accessing multiple elements: `a[start:end]`.

**Relationship-Based S

In [ ]:
#design a chat CLI

def chat_cli():
    messages = [{"role": "system", "content": updated_prompt}] #FIX: only asking about array in next step when you have more topics
    print("Hello! I am your CS Study Tutor. Let's start studying! Are you ready?")
    while True:
        user_input = input()
        if user_input == "quit" or user_input == "exit":
            break
        calculate_peh1_value(user_profile)
        messages.append({"role": "user", "content": user_input})
        
        llm_call = client.chat.completions.create(
            model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
            messages=messages)
        response_text = llm_call.choices[0].message.content
        print("AI", response_text)

        messages.append({"role": "assistant", "content": response_text})

chat_cli()


Hello! I am your CS Study Tutor. Let's start studying! Are you ready?
AI ### Notes on Arrays in Python

1. An array is a collection of items stored at contiguous memory locations. 

2. It's a data structure that can hold multiple values of the same type.

3. **Key Points**:
   - Arrays are primarily used to store lists of data.
   - Python does not have a built-in array data type like other languages; however, the `array` module and `numpy` library provide array functionality.
   - The `array` module allows you to create an array using the syntax: `import array` followed by `my_array = array.array('typecode', [values])`.

### Using the array Module
- Install the module if necessary.
- Import the array module.
- Define the type of array, e.g., `'i'` for integers or `'d'` for doubles.
- Example:
  ```python
  import array
  my_array = array.array('i', [1, 2, 3, 4, 5])
  ```

### Numpy Arrays
- Numpy is a powerful library for numerical computations in Python.
- Install Numpy using: `pip i

In [44]:
#without bayesian theorem approach 
without_theorem_prompt = f'''
You're a Computer Science note maker. 

Now generate notes on Array using python coding language in 300 words. 
'''

# Send the full prompt to OpenAI
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
    messages=[
        {"role": "user", "content": without_theorem_prompt},
    ]
)

# Print the response text
response_text = response.choices[0].message.content
print(response_text)



# Arrays in Python

### Introduction
An array is a data structure that stores a collection of elements, usually of the same data type, in a contiguous block of memory. Arrays allow efficient access and manipulation of elements using indices.

### Creating Arrays
Python doesn't have a built-in array data type like some other languages (e.g., C, Java). However, you can use two main approaches for working with arrays:

1. **Lists**: The most common Python data structure that can be used as an array. Lists are flexible but not as efficient in terms of speed and memory as true arrays.
   ```python
   # Creating a list
   my_list = [1, 2, 3, 4, 5]
   ```

2. **Array Module**: Provides a true array data structure but is limited to basic types and operations.
   ```python
   import array
   # Creating an array of integers
   my_array = array.array('i', [1, 2, 3, 4, 5])
   ```

3. **NumPy Arrays**: NumPy is a powerful library for numerical computing in Python. NumPy arrays provide more function